In [0]:
WITH
first_timers AS (
    SELECT cc_user_profile_id
    FROM cpx_dataanalytics_gold.marketing.mkt_subscription_events
    WHERE event_type = 'subscription_created' AND cc_user_profile_id IS NOT NULL
    GROUP BY cc_user_profile_id
    HAVING COUNT(*) = 1
),
upgrades_took_effect AS (
    SELECT DISTINCT cc_user_profile_id
    FROM cpx_dataanalytics_gold.marketing.mkt_subscription_events
    WHERE event_type = 'subscription_cancelled' AND plan_id = 'monthly1' AND changed = TRUE
      AND reporting_datetime BETWEEN '2025-11-28' AND '2025-12-01'
),
a_first_time AS (
    SELECT DISTINCT e.cc_user_profile_id
    FROM cpx_dataanalytics_gold.marketing.mkt_subscription_events AS e
    INNER JOIN first_timers AS f ON e.cc_user_profile_id = f.cc_user_profile_id
    LEFT ANTI JOIN upgrades_took_effect AS ute ON e.cc_user_profile_id = ute.cc_user_profile_id
    WHERE e.event_type = 'subscription_created' AND e.plan_id = 'annual1'
      AND e.reporting_datetime BETWEEN '2025-11-28' AND '2025-12-01'
),
a_returning AS (
    SELECT DISTINCT e.cc_user_profile_id
    FROM cpx_dataanalytics_gold.marketing.mkt_subscription_events AS e
    LEFT ANTI JOIN first_timers AS f ON e.cc_user_profile_id = f.cc_user_profile_id
    LEFT ANTI JOIN upgrades_took_effect AS ute ON e.cc_user_profile_id = ute.cc_user_profile_id
    WHERE e.event_type = 'subscription_created' AND e.plan_id = 'annual1'
      AND e.reporting_datetime BETWEEN '2025-11-28' AND '2025-12-01'
),
a_total_signups AS (
    SELECT DISTINCT e.cc_user_profile_id
    FROM cpx_dataanalytics_gold.marketing.mkt_subscription_events AS e
    LEFT ANTI JOIN upgrades_took_effect AS ute ON e.cc_user_profile_id = ute.cc_user_profile_id
    WHERE e.event_type = 'subscription_created' AND e.plan_id = 'annual1'
      AND e.reporting_datetime BETWEEN '2025-11-28' AND '2025-12-01'
),
a_exits AS (
    SELECT DISTINCT cc_user_profile_id
    FROM cpx_dataanalytics_gold.marketing.mkt_subscription_events
    WHERE event_type = 'subscription_cancelled' AND plan_id = 'annual1' AND cc_user_profile_id IS NOT NULL
      AND changed IS NOT TRUE
      AND reporting_datetime BETWEEN '2025-11-28' AND '2025-12-01'
),
annual_direct_signup_starts AS (
    SELECT cc_user_profile_id, MIN(CAST(reporting_datetime AS DATE)) AS start_date
    FROM cpx_dataanalytics_gold.marketing.mkt_subscription_events
    WHERE event_type = 'subscription_created' AND plan_id = 'annual1' AND reporting_datetime IS NOT NULL
    GROUP BY cc_user_profile_id
),
annual_upgrade_starts AS (
    SELECT cc_user_profile_id, MIN(CAST(reporting_datetime AS DATE)) AS start_date
    FROM cpx_dataanalytics_gold.marketing.mkt_subscription_events
    WHERE event_type = 'subscription_cancelled' AND plan_id = 'monthly1' AND changed = TRUE AND reporting_datetime IS NOT NULL
    GROUP BY cc_user_profile_id
),
annual_starts AS (
    SELECT cc_user_profile_id, MIN(start_date) AS start_date
    FROM (SELECT * FROM annual_direct_signup_starts UNION ALL SELECT * FROM annual_upgrade_starts)
    GROUP BY cc_user_profile_id
),
annual_payments_window AS (
    SELECT DISTINCT cc_user_profile_id, reporting_datetime
    FROM cpx_dataanalytics_gold.marketing.mkt_subscription_events
    WHERE event_type = 'payment_succeeded' AND plan_id = 'annual1'
      AND reporting_datetime BETWEEN '2025-11-28' AND '2025-12-01'
),
annual_renewed AS (
    SELECT DISTINCT p.cc_user_profile_id
    FROM annual_payments_window AS p
    INNER JOIN annual_starts AS s ON p.cc_user_profile_id = s.cc_user_profile_id
    LEFT ANTI JOIN a_total_signups AS sg ON sg.cc_user_profile_id = p.cc_user_profile_id
    LEFT ANTI JOIN a_exits AS ex ON ex.cc_user_profile_id = p.cc_user_profile_id
    LEFT ANTI JOIN upgrades_took_effect AS ute ON ute.cc_user_profile_id = p.cc_user_profile_id
    WHERE DATEDIFF(CAST(p.reporting_datetime AS DATE), s.start_date) >= 360
      AND (MOD(DATEDIFF(CAST(p.reporting_datetime AS DATE), s.start_date), 365) <= 5
           OR MOD(DATEDIFF(CAST(p.reporting_datetime AS DATE), s.start_date), 365) >= 360)
),
prev_window_snapshot AS (
    SELECT DISTINCT cc_user_profile_id, plan_id
    FROM cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot
    WHERE snapshot_date_id = '20251127'
),
g_purchasers AS (
    SELECT DISTINCT purchaser_id
    FROM cpx_dataanalytics_silver.cineclub.gifting
    WHERE CAST(from_utc_timestamp(CAST(purchase_date AS TIMESTAMP), 'America/New_York') AS DATE)
          BETWEEN '2025-11-28' AND '2025-12-01'
),
g_redeemers AS (
    SELECT DISTINCT redeemer_id
    FROM cpx_dataanalytics_silver.cineclub.gifting
    WHERE is_redeemed = TRUE
      AND CAST(from_utc_timestamp(CAST(redemption_date AS TIMESTAMP), 'America/New_York') AS DATE)
          BETWEEN '2025-11-28' AND '2025-12-01'
),
gift_purchasers_net_new AS (
    SELECT t.purchaser_id AS cc_user_profile_id FROM g_purchasers t
    LEFT ANTI JOIN prev_window_snapshot p ON t.purchaser_id = p.cc_user_profile_id
),
gift_redeemers_net_new AS (
    SELECT t.redeemer_id AS cc_user_profile_id FROM g_redeemers t
    LEFT ANTI JOIN prev_window_snapshot p ON t.redeemer_id = p.cc_user_profile_id
),
bf25_all_profile_ids AS (
    SELECT cc_user_profile_id, 'CPA' AS cost_bucket FROM a_first_time
    UNION ALL SELECT cc_user_profile_id, 'CPA' FROM a_returning
    UNION ALL SELECT cc_user_profile_id, 'CPA' FROM upgrades_took_effect
    UNION ALL SELECT cc_user_profile_id, 'CPR' FROM annual_renewed
    UNION ALL SELECT cc_user_profile_id, 'CPA' FROM gift_purchasers_net_new
    UNION ALL SELECT cc_user_profile_id, 'CPA' FROM gift_redeemers_net_new
),
cde_lookup_window AS (
    SELECT
        cc_user_profile_id,
        cde_id,
        ROW_NUMBER() OVER (PARTITION BY cc_user_profile_id ORDER BY snapshot_date_id DESC) AS rn
    FROM cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot
    WHERE snapshot_date_id BETWEEN '20251127' AND '20251201'
),
bf25_population AS (
    -- Note: a cde_id could theoretically appear in both CPA and CPR if the same person
    -- both signed up AND renewed in this window (edge case, e.g. someone who joined then
    -- immediately had a separate renewal event). Taking DISTINCT cost_bucket per cde_id.
    SELECT DISTINCT CAST(cl.cde_id AS STRING) AS cde_id, p.cost_bucket
    FROM bf25_all_profile_ids p
    JOIN cde_lookup_window cl ON p.cc_user_profile_id = cl.cc_user_profile_id
    WHERE cl.rn = 1
),

post_base_trans AS (
    SELECT t.CDE_ID AS cde_id, d.datefull, t.CineClubFLAG, t.SCENEFLAG, t.Quantity
    FROM cpx_dataanalytics_gold.customer360.c360_f_transaction t
    JOIN cpx_dataanalytics_gold.customer360.cpx_d_location loc ON t.LocationId = loc.LocationID
    JOIN cpx_dataanalytics_gold.customer360.cpx_d_date d ON t.VisitDateId = d.dateid
    WHERE loc.LC_Location_Type_Description = 'Cineplex Theatre'
      AND t.CDE_ID IS NOT NULL
      AND d.datefull BETWEEN DATE'2026-01-01' AND DATE'2026-08-01'
      AND (t.SCENEFLAG = 'Yes' OR t.CineClubFLAG = 'Yes')
),
post_segmented AS (
    SELECT
        bt.cde_id, bt.datefull,
        CASE
            WHEN bt.SCENEFLAG = 'Yes' AND bt.CineClubFLAG IS DISTINCT FROM 'Yes' THEN 'Scene'
            WHEN bt.CineClubFLAG = 'Yes' AND bp.cost_bucket = 'CPA' THEN 'CC_Acquired_Offer_CPA'
            WHEN bt.CineClubFLAG = 'Yes' AND bp.cost_bucket = 'CPR' THEN 'CC_Acquired_Offer_CPR'
            WHEN bt.CineClubFLAG = 'Yes' THEN 'CC_Not_Acquired_Offer'
        END AS segment,
        bt.Quantity
    FROM post_base_trans bt
    LEFT JOIN bf25_population bp ON CAST(bt.cde_id AS STRING) = bp.cde_id
),
population_lock AS (
    SELECT DISTINCT CAST(cde_id AS STRING) AS cde_id, segment
    FROM post_segmented
    WHERE segment IS NOT NULL
    GROUP BY cde_id, segment
    HAVING SUM(Quantity) > 0
),

all_base_trans AS (
    SELECT
        CAST(t.CDE_ID AS STRING) AS cde_id,
        d.datefull,
        t.TicketFLAG,
        t.ConcessionFLAG,
        t.Quantity,
        t.Net_Revenue,
        t.CineClub_Discount_Type_ID AS discount_type_id,
        s.Auditorium_Experience_ID,
        s.Performance_Experience_ID,
        s.Is_DBOXflag,
        CASE
            WHEN d.datefull BETWEEN DATE'2025-01-01' AND DATE'2025-08-01' THEN 'Pre-Offer'
            WHEN d.datefull BETWEEN DATE'2026-01-01' AND DATE'2026-08-01' THEN 'Post-Offer'
        END AS period
    FROM cpx_dataanalytics_gold.customer360.c360_f_transaction t
    JOIN cpx_dataanalytics_gold.customer360.cpx_d_location loc ON t.LocationId = loc.LocationID
    JOIN cpx_dataanalytics_gold.customer360.cpx_d_date d ON t.VisitDateId = d.dateid
    LEFT JOIN cpx_dataanalytics_gold.customer360.cpx_d_session s ON t.sessionskey = s.sessionskey
    WHERE loc.LC_Location_Type_Description = 'Cineplex Theatre'
      AND t.CDE_ID IS NOT NULL
      AND (d.datefull BETWEEN DATE'2025-01-01' AND DATE'2025-08-01'
           OR d.datefull BETWEEN DATE'2026-01-01' AND DATE'2026-08-01')
),
segmented_all AS (
    SELECT bt.*, pl.segment
    FROM all_base_trans bt
    INNER JOIN population_lock pl ON bt.cde_id = pl.cde_id
),
format_classified AS (
    SELECT
        *,
        CASE
            WHEN Auditorium_Experience_ID = 2      THEN 'AVX'
            WHEN Auditorium_Experience_ID = 3      THEN 'IMAX'
            WHEN Auditorium_Experience_ID = 4      THEN 'VIP'
            WHEN Auditorium_Experience_ID IN (5,6) THEN 'ALT'
            WHEN Auditorium_Experience_ID = 7      THEN 'FourDX'
            WHEN Auditorium_Experience_ID = 8      THEN 'ScreenX'
            WHEN Performance_Experience_ID = 2     THEN 'ThreeD'
            ELSE 'Regular'
        END AS final_format
    FROM segmented_all
),
visit_grain AS (
    SELECT
        period, segment, cde_id, datefull,
        SUM(CASE WHEN ConcessionFLAG = 'Yes' THEN Net_Revenue ELSE 0 END) AS concession_spend,
        SUM(CASE WHEN ConcessionFLAG = 'Yes' THEN Quantity ELSE 0 END) AS concession_qty,
        CASE WHEN SUM(Quantity) > 0 THEN 1 ELSE 0 END AS is_visit
    FROM format_classified
    GROUP BY period, segment, cde_id, datefull
),
visit_agg AS (
    SELECT
        period, segment,
        COUNT(DISTINCT CASE WHEN is_visit = 1 THEN cde_id END) AS distinct_members,
        SUM(is_visit) AS total_visits,
        ROUND(SUM(is_visit) * 1.0 / COUNT(DISTINCT CASE WHEN is_visit = 1 THEN cde_id END), 2) AS avg_visits_per_member,
        ROUND(SUM(CASE WHEN is_visit = 1 THEN concession_spend ELSE 0 END) / COUNT(DISTINCT CASE WHEN is_visit = 1 THEN cde_id END), 2) AS avg_concession_spend_per_member,
        ROUND(SUM(concession_qty) * 1.0 / NULLIF(SUM(is_visit), 0), 2) AS concession_incidence
    FROM visit_grain
    GROUP BY period, segment
),
ticket_agg AS (
    SELECT
        period, segment,
        SUM(Quantity) AS total_tickets,
        SUM(Net_Revenue) AS ticket_spend,
        SUM(CASE WHEN discount_type_id = 6 THEN Quantity ELSE 0 END) AS tickets_redeemed,
        SUM(CASE WHEN discount_type_id = 5 THEN Quantity ELSE 0 END) AS tickets_companion,
        SUM(CASE WHEN final_format != 'Regular' OR Is_DBOXflag = 'Yes' THEN Quantity ELSE 0 END) AS tickets_premium
    FROM format_classified
    WHERE TicketFLAG = 'Yes'
    GROUP BY period, segment
),
concession_total AS (
    SELECT period, segment, SUM(concession_spend) AS total_concession_spend
    FROM visit_grain
    GROUP BY period, segment
)

SELECT
    va.period,
    va.segment,
    va.distinct_members,
    va.total_visits,
    va.avg_visits_per_member,
    ta.total_tickets,
    ROUND(ta.total_tickets * 1.0 / va.distinct_members, 2) AS tickets_per_member,
    ROUND(ta.tickets_redeemed * 1.0 / va.distinct_members, 2) AS tickets_redeemed_per_member,
    ROUND(ta.tickets_companion * 1.0 / va.distinct_members, 2) AS tickets_companion_per_member,
    va.avg_concession_spend_per_member,
    va.concession_incidence,
    ROUND(ta.tickets_premium * 1.0 / va.distinct_members, 2) AS premium_tickets_per_member,
    ROUND(ta.ticket_spend / va.distinct_members, 2) AS avg_ticket_spend_per_member,
    ROUND((ta.ticket_spend + ct.total_concession_spend) / va.distinct_members, 2) AS avg_total_revenue_per_member
FROM visit_agg va
LEFT JOIN ticket_agg ta ON va.period = ta.period AND va.segment = ta.segment
LEFT JOIN concession_total ct ON va.period = ct.period AND va.segment = ct.segment
ORDER BY segment, CASE va.period WHEN 'Pre-Offer' THEN 0 ELSE 1 END;